In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="14ahkZ61qf2-80W8eXMwK8fDSCGO0G0Dq", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/05_00_intro.mp3"))


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_02_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# Hybrid Memory Capstone: Building a Production-Ready System

*Part 5 of the Vizuara series on Memory Architectures for LLM Applications*

**Estimated time: 65 minutes**

We have now built four memory systems: conversation buffer, sliding window, summary memory, entity memory, and vector-backed memory. Each one excels at something different — the buffer remembers everything but is expensive, the window is cheap but forgetful, summaries preserve themes but lose details, entities capture precise facts but miss general context, and vectors enable semantic retrieval but cannot detect contradictions. In this capstone notebook, we will combine all four into a single **hybrid memory architecture** that uses each subsystem for what it does best. We will implement a token budget allocator, build the complete system, and run it through a rigorous evaluation comparing it against every individual approach.

## 1. Why Does This Matter?

Think about how human memory actually works. You do not have a single memory system — you have several that work together:

- **Working memory** (like our sliding window): holds the last few seconds of conversation in full detail
- **Semantic memory** (like our entity store): stores facts — "Paris is the capital of France"
- **Episodic memory** (like our vector store): stores experiences — "I visited Paris last summer and it rained"
- **Gist memory** (like our summary): holds the overall narrative — "We had a good vacation in Europe"

These systems do not compete — they **complement** each other. When someone asks "How was your trip to France?", your brain pulls from all of them simultaneously: the gist gives the overall tone, episodic memory retrieves specific moments, semantic memory provides factual details, and working memory maintains the flow of the current conversation.

A hybrid memory architecture does the same for an LLM. The key challenge is **budget allocation** — dividing a finite context window across multiple memory types to maximize the quality of the LLM's response.

In this notebook, we will:
- Design a **token budget allocator** that divides the context window across memory types
- Build the **complete hybrid system** combining all four memory subsystems
- Implement a **context assembler** that merges outputs from all subsystems
- Run a **comprehensive evaluation** comparing hybrid vs individual approaches
- Build a **production-ready chatbot** powered by the hybrid memory

Let us begin.

In [ ]:
#@title 🎧 Listen: Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_03_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition

### The Budget Allocation Problem

Suppose our LLM has an 8,000-token context window. After reserving tokens for the system prompt (500) and the response (2,000), we have 5,500 tokens for memory. How should we divide them?

```
Available: 5,500 tokens for memory

Option A: All to recent messages
  [Recent: 5,500 tokens] — great continuity, no old context

Option B: All to vector retrieval
  [Retrieved: 5,500 tokens] — great recall, no continuity

Option C: Hybrid allocation
  [Summary:   300 tokens] — conversation overview
  [Entities:  200 tokens] — precise facts
  [Retrieved: 1,000 tokens] — semantic recall
  [Recent:    2,000 tokens] — conversation flow
  [Buffer:    2,000 tokens] — spare capacity
```

Option C is the most robust. It ensures the LLM has the big picture (summary), key facts (entities), relevant older context (vector), and smooth conversational flow (recent). No single failure mode can cripple the system.

### The Priority Stack

Not all memory types are equally important for every query. A factual question ("What is my budget?") needs entity memory more than a summary. A broad question ("What have we discussed so far?") needs the summary more than entities. A follow-up question ("Can you elaborate on that?") needs recent messages most of all.

The hybrid system should be **adaptive** — adjusting budget allocation based on the query type. We will implement this with a simple priority stack:

1. **Always include**: System prompt + recent messages (minimum 3 turns)
2. **High priority**: Entities matching the query keywords
3. **Medium priority**: Semantically retrieved vector memories
4. **Low priority**: Running summary (only if budget remains)

This ensures that the most useful context always makes it into the prompt.

In [ ]:
#@title 🎧 Listen: Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_04_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics

### Token Budget Equation

The total context window must accommodate all components:

$$T_{\text{total}} = T_{\text{system}} + T_{\text{summary}} + T_{\text{entities}} + T_{\text{retrieved}} + T_{\text{recent}} + T_{\text{response}}$$

Rearranging for the available memory budget:

$$B = T_{\text{total}} - T_{\text{system}} - T_{\text{response}}$$

We then allocate $B$ across memory types using allocation weights $w_i$ that sum to 1:

$$T_i = w_i \cdot B$$

**Worked example:** With $T_{\text{total}} = 8{,}000$, $T_{\text{system}} = 500$, $T_{\text{response}} = 2{,}000$:

$$B = 8{,}000 - 500 - 2{,}000 = 5{,}500 \text{ tokens}$$

With weights $w_{\text{summary}} = 0.06$, $w_{\text{entities}} = 0.04$, $w_{\text{retrieved}} = 0.36$, $w_{\text{recent}} = 0.54$:

| Component | Weight | Tokens |
|-----------|--------|--------|
| Summary | 0.06 | 330 |
| Entities | 0.04 | 220 |
| Retrieved | 0.36 | 1,980 |
| Recent | 0.54 | 2,970 |
| **Total** | **1.00** | **5,500** |

### Quality Score

To evaluate the hybrid system, we define a composite **quality score** that measures performance across multiple dimensions:

$$Q = \alpha_1 \cdot R_{\text{fact}} + \alpha_2 \cdot R_{\text{semantic}} + \alpha_3 \cdot R_{\text{continuity}} + \alpha_4 \cdot (1 - C_{\text{norm}})$$

where:
- $R_{\text{fact}}$: Factual recall (entity-based, 0-1)
- $R_{\text{semantic}}$: Semantic recall (vector-based, 0-1)
- $R_{\text{continuity}}$: Conversation continuity (recent turns coverage, 0-1)
- $C_{\text{norm}}$: Normalized cost (tokens used / budget, 0-1)
- $\alpha_i$: Importance weights (sum to 1)

**Worked example:** With $\alpha_1 = 0.3$, $\alpha_2 = 0.3$, $\alpha_3 = 0.2$, $\alpha_4 = 0.2$:

$$Q = 0.3 \times 0.85 + 0.3 \times 0.90 + 0.2 \times 1.0 + 0.2 \times (1 - 0.65) = 0.255 + 0.270 + 0.200 + 0.070 = 0.795$$

A quality score of 0.795 means the system does well on fact recall and semantic retrieval, has perfect continuity, and uses 65% of the available budget.

In [ ]:
#@title 🎧 Code Walkthrough: Env Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_05_env_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Setting Up Our Environment

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy matplotlib tiktoken

In [ ]:
import numpy as np
import faiss
import matplotlib.pyplot as plt
import tiktoken
import time
import re
import json
from typing import List, Dict, Tuple, Optional, Set
from collections import defaultdict, deque
from dataclasses import dataclass, field

ENCODER = tiktoken.encoding_for_model("gpt-4")

def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))

# Load embedding model
print("Loading embedding model...")
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Model loaded! Dim: {EMBED_DIM}")

In [ ]:
#@title 🎧 Listen: Subsystems Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_06_subsystems_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Let's Build It — Component by Component

### 4.1 The Individual Memory Subsystems

First, let us define clean, minimal versions of each memory type that we will plug into the hybrid system.

In [ ]:
#@title 🎧 Code Walkthrough: Sliding Window Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_07_sliding_window_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Sliding Window ---
class SlidingWindow:
    """Recent conversation window."""
    def __init__(self, max_tokens: int = 2000):
        self.max_tokens = max_tokens
        self.messages: List[Dict] = []

    def add(self, role: str, content: str):
        tokens = count_tokens(content)
        self.messages.append({
            "role": role, "content": content, "tokens": tokens,
        })

    def get_context(self) -> List[Dict]:
        """Return recent messages that fit in budget."""
        result = []
        total = 0
        for msg in reversed(self.messages):
            if total + msg["tokens"] > self.max_tokens:
                break
            result.insert(0, msg)
            total += msg["tokens"]
        return result

    def get_tokens(self) -> int:
        ctx = self.get_context()
        return sum(m["tokens"] for m in ctx)

print("SlidingWindow defined!")

In [ ]:
#@title 🎧 Code Walkthrough: Summary Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_08_summary_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Summary Memory ---
class SummaryEngine:
    """Maintains a running summary of the conversation."""
    def __init__(self, max_tokens: int = 300):
        self.max_tokens = max_tokens
        self.summary = ""
        self.summary_tokens = 0
        self.all_messages: List[Dict] = []
        self.last_summarized = 0

    def add(self, role: str, content: str):
        self.all_messages.append({"role": role, "content": content})

    def update_summary(self, recent_start: int):
        """Summarize messages before recent_start."""
        if recent_start <= self.last_summarized:
            return
        old = self.all_messages[self.last_summarized:recent_start]
        if not old:
            return

        # Rule-based summarization
        text = " ".join(m["content"] for m in old)
        entities = set()
        for word in text.split():
            cleaned = word.strip(".,!?\"'()[]")
            if (cleaned and cleaned[0].isupper()
                    and len(cleaned) > 1 and cleaned not in
                    {"The", "This", "That", "I", "We", "You",
                     "What", "How", "User", "Assistant", "Yes",
                     "No", "Great", "Sure", "Hello"}):
                entities.add(cleaned)

        facts = []
        for m in re.finditer(r'\$[\d,]+', text):
            facts.append(f"budget {m.group()}")
        for m in re.finditer(
            r'((?:January|February|March|April|May|June|'
            r'July|August|September|October|November|'
            r'December)\s+\d{1,2})', text
        ):
            facts.append(f"deadline {m.group()}")

        parts = []
        if self.summary:
            parts.append(self.summary)
        if entities:
            parts.append(f"Mentioned: {', '.join(sorted(entities)[:10])}")
        if facts:
            parts.append(f"Key facts: {'; '.join(facts[:5])}")

        self.summary = ". ".join(parts)
        # Truncate to budget
        words = self.summary.split()
        max_words = int(self.max_tokens * 0.75)
        if len(words) > max_words:
            self.summary = " ".join(words[:max_words]) + "..."

        self.summary_tokens = count_tokens(self.summary)
        self.last_summarized = recent_start

    def get_context(self) -> str:
        return self.summary

    def get_tokens(self) -> int:
        return self.summary_tokens

print("SummaryEngine defined!")

In [ ]:
#@title 🎧 Code Walkthrough: Entity Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_09_entity_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Entity Store ---
class EntityStore:
    """Structured fact storage with keyword retrieval."""
    def __init__(self, max_tokens: int = 200):
        self.max_tokens = max_tokens
        self.entities: Dict[str, Dict] = {}

    def extract_and_store(self, text: str, turn: int):
        """Extract entities from text and store them."""
        lower = text.lower()

        # Names
        for m in re.finditer(
            r"(?:my name is|i am|i'm) (\w+)", text, re.I
        ):
            name = m.group(1)
            if name not in ("a", "an", "the", "not", "very", "also"):
                self.entities["user_name"] = {
                    "value": name, "turn": turn,
                    "keywords": {"name", name.lower()},
                }

        # Dollar amounts
        for m in re.finditer(r'\$[\d,]+', text):
            self.entities["budget"] = {
                "value": m.group(), "turn": turn,
                "keywords": {"budget", "money", "cost", "spend",
                             m.group().replace("$", "").replace(",", "")},
            }

        # Dates
        for m in re.finditer(
            r'((?:January|February|March|April|May|June|'
            r'July|August|September|October|November|'
            r'December)\s+\d{1,2})', text
        ):
            self.entities["deadline"] = {
                "value": m.group(), "turn": turn,
                "keywords": {"deadline", "date", "due", "when",
                             m.group().lower()},
            }

        # Team members
        for m in re.finditer(
            r'(\w+)\s+(?:handles?|does|works on)\s+(\w+)',
            text, re.I
        ):
            name = m.group(1)
            role = m.group(2)
            if name[0].isupper() and name not in {"The", "This"}:
                self.entities[f"team_{name.lower()}"] = {
                    "value": f"{name}: {role}",
                    "turn": turn,
                    "keywords": {"team", name.lower(), role.lower()},
                }

        # Numbers with units
        for m in re.finditer(
            r'(\d+)\s+(people|members|weeks|days|percent|%)',
            text
        ):
            self.entities[f"count_{m.group(2)}"] = {
                "value": f"{m.group(1)} {m.group(2)}",
                "turn": turn,
                "keywords": {m.group(2), "count", "number"},
            }

        # Technology
        techs = {
            "python": "language", "fastapi": "framework",
            "react": "frontend", "postgresql": "database",
            "docker": "deployment", "kubernetes": "orchestration",
            "oauth": "auth", "pytest": "testing",
            "prometheus": "monitoring", "grafana": "dashboards",
        }
        for tech, category in techs.items():
            if tech in lower:
                self.entities[f"tech_{tech}"] = {
                    "value": f"{tech.capitalize()} ({category})",
                    "turn": turn,
                    "keywords": {"technology", tech, category},
                }

    def retrieve(self, query: str, max_tokens: int = None) -> str:
        """Retrieve relevant entities as formatted text."""
        if max_tokens is None:
            max_tokens = self.max_tokens

        query_words = set(query.lower().split()) - {
            "the", "a", "an", "is", "are", "what", "how",
            "my", "i", "we", "you", "to", "for", "of",
            "about", "can", "do", "did",
        }

        scored = []
        for key, info in self.entities.items():
            overlap = len(query_words & info["keywords"])
            if overlap > 0:
                scored.append((key, info, overlap))

        scored.sort(key=lambda x: x[2], reverse=True)

        lines = []
        total_tokens = 0
        for key, info, _ in scored:
            line = f"- {key}: {info['value']}"
            line_tokens = count_tokens(line)
            if total_tokens + line_tokens > max_tokens:
                break
            lines.append(line)
            total_tokens += line_tokens

        return "\n".join(lines)

    def get_tokens(self, query: str = "general") -> int:
        text = self.retrieve(query)
        return count_tokens(text) if text else 0

    @property
    def size(self):
        return len(self.entities)

print("EntityStore defined!")

In [ ]:
#@title 🎧 Code Walkthrough: Vector Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_10_vector_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Vector Memory ---
class VectorMemory:
    """FAISS-backed semantic retrieval."""
    def __init__(self, max_tokens: int = 1000, top_k: int = 5):
        self.max_tokens = max_tokens
        self.top_k = top_k
        self.index = faiss.IndexFlatIP(EMBED_DIM)
        self.messages: List[Dict] = []

    def add(self, content: str, role: str):
        emb = embed_model.encode([content]).astype("float32")
        faiss.normalize_L2(emb)
        self.index.add(emb)
        self.messages.append({
            "content": content, "role": role,
            "tokens": count_tokens(content),
        })

    def retrieve(
        self, query: str, exclude_recent: Set[str] = None,
    ) -> List[Tuple[Dict, float]]:
        """Retrieve top-k semantically similar messages."""
        if self.index.ntotal == 0:
            return []

        q_emb = embed_model.encode([query]).astype("float32")
        faiss.normalize_L2(q_emb)

        k = min(self.top_k * 2, self.index.ntotal)
        scores, indices = self.index.search(q_emb, k)

        results = []
        total_tokens = 0
        exclude = exclude_recent or set()

        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or idx >= len(self.messages):
                continue
            msg = self.messages[idx]
            if msg["content"] in exclude:
                continue
            if total_tokens + msg["tokens"] > self.max_tokens:
                continue
            results.append((msg, float(score)))
            total_tokens += msg["tokens"]
            if len(results) >= self.top_k:
                break

        return results

    def format_results(self, results: List[Tuple]) -> str:
        return "\n".join(
            f"[{score:.2f}] {msg['role']}: {msg['content']}"
            for msg, score in results
        )

    @property
    def size(self):
        return len(self.messages)

print("VectorMemory defined!")

In [ ]:
#@title 🎧 Listen: Allocator Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_11_allocator_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.2 The Token Budget Allocator

This is the heart of the hybrid system — it decides how many tokens each subsystem gets.

In [ ]:
#@title 🎧 Code Walkthrough: Allocator Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_12_allocator_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class BudgetAllocator:
    """Allocates context window tokens across memory types.

    Supports fixed allocation and adaptive allocation
    based on query characteristics.
    """

    def __init__(
        self,
        total_tokens: int = 8000,
        system_tokens: int = 500,
        response_reserve: int = 2000,
    ):
        self.total = total_tokens
        self.system = system_tokens
        self.reserve = response_reserve
        self.budget = total_tokens - system_tokens - response_reserve

    def allocate_fixed(
        self,
        weights: Dict[str, float] = None,
    ) -> Dict[str, int]:
        """Fixed allocation using predefined weights."""
        if weights is None:
            weights = {
                "summary": 0.06,
                "entities": 0.04,
                "retrieved": 0.36,
                "recent": 0.54,
            }

        assert abs(sum(weights.values()) - 1.0) < 0.01, \
            "Weights must sum to 1"

        allocation = {}
        for key, weight in weights.items():
            allocation[key] = int(self.budget * weight)

        # Assign any remainder to recent
        remainder = self.budget - sum(allocation.values())
        allocation["recent"] += remainder

        return allocation

    def allocate_adaptive(
        self,
        query: str,
        n_entities: int,
        n_messages: int,
    ) -> Dict[str, int]:
        """Adaptive allocation based on query type.

        Rules:
        - Factual queries (who/what/when/how much) ->
          boost entities
        - Broad queries (summarize/overview/tell me about) ->
          boost summary + retrieved
        - Follow-up queries (elaborate/continue/more) ->
          boost recent
        - Default: balanced allocation
        """
        lower = query.lower()

        if any(w in lower for w in [
            "how much", "budget", "cost", "name", "who",
            "when", "deadline", "date",
        ]):
            # Factual query -> boost entities
            weights = {
                "summary": 0.05,
                "entities": 0.15,
                "retrieved": 0.30,
                "recent": 0.50,
            }
        elif any(w in lower for w in [
            "summarize", "overview", "tell me about",
            "what have we", "recap",
        ]):
            # Broad query -> boost summary + retrieved
            weights = {
                "summary": 0.15,
                "entities": 0.05,
                "retrieved": 0.40,
                "recent": 0.40,
            }
        elif any(w in lower for w in [
            "elaborate", "continue", "more about that",
            "go on", "explain further",
        ]):
            # Follow-up -> boost recent
            weights = {
                "summary": 0.03,
                "entities": 0.02,
                "retrieved": 0.15,
                "recent": 0.80,
            }
        else:
            # Balanced default
            weights = {
                "summary": 0.06,
                "entities": 0.04,
                "retrieved": 0.36,
                "recent": 0.54,
            }

        return self.allocate_fixed(weights)

print("BudgetAllocator defined!")

In [ ]:
#@title 🎧 Code Walkthrough: Test Allocator
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_13_test_allocator.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Let us test the allocator:

In [ ]:
allocator = BudgetAllocator(
    total_tokens=8000,
    system_tokens=500,
    response_reserve=2000,
)

print(f"Total budget: {allocator.budget} tokens\n")

# Test different query types
test_queries = [
    "What is my budget?",          # Factual
    "Give me an overview of the project.",  # Broad
    "Can you elaborate on that last point?", # Follow-up
    "What tech stack should we use?",  # Default
]

for query in test_queries:
    alloc = allocator.allocate_adaptive(query, 10, 20)
    print(f"Query: \"{query}\"")
    for key, tokens in alloc.items():
        pct = tokens / allocator.budget * 100
        bar = "=" * int(pct / 2)
        print(f"  {key:>10}: {tokens:>5} tokens ({pct:4.1f}%) {bar}")
    print()

In [ ]:
#@title 🎧 Listen: Hybrid Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_14_hybrid_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.3 The Hybrid Memory System

Now let us wire everything together into the complete hybrid system.

In [ ]:
#@title 🎧 Code Walkthrough: Hybrid Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_15_hybrid_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class HybridMemory:
    """Production-ready hybrid memory system.

    Combines sliding window, summary, entity store,
    and vector memory with adaptive budget allocation.
    """

    def __init__(
        self,
        total_tokens: int = 8000,
        system_prompt: str = "",
        response_reserve: int = 2000,
        recent_turns: int = 5,
        vector_top_k: int = 5,
    ):
        self.system_prompt = system_prompt
        system_tokens = count_tokens(system_prompt)

        # Budget allocator
        self.allocator = BudgetAllocator(
            total_tokens=total_tokens,
            system_tokens=system_tokens,
            response_reserve=response_reserve,
        )

        # Memory subsystems
        self.window = SlidingWindow(max_tokens=3000)
        self.summary = SummaryEngine(max_tokens=400)
        self.entities = EntityStore(max_tokens=300)
        self.vectors = VectorMemory(
            max_tokens=2000, top_k=vector_top_k,
        )

        # State
        self.all_messages: List[Dict] = []
        self.turn_count = 0

        # Tracking
        self.allocation_log: List[Dict] = []
        self.context_token_log: List[int] = []
        self.subsystem_token_log: List[Dict] = []

    def add_message(self, role: str, content: str):
        """Process a new message through all subsystems."""
        tokens = count_tokens(content)
        self.all_messages.append({
            "role": role, "content": content, "tokens": tokens,
        })

        if role == "user":
            self.turn_count += 1

        # Feed to all subsystems
        self.window.add(role, content)
        self.summary.add(role, content)
        self.entities.extract_and_store(content, self.turn_count)
        self.vectors.add(content, role)

        # Update summary periodically
        if self.turn_count % 5 == 0 and self.turn_count > 0:
            recent_start = max(
                0, len(self.all_messages) - 10
            )
            self.summary.update_summary(recent_start)

    def build_context(self, query: str) -> Dict:
        """Build the full context for an LLM prompt.

        Returns dict with 'context' (string), 'tokens',
        and 'allocation' (token breakdown).
        """
        # Get adaptive allocation
        allocation = self.allocator.allocate_adaptive(
            query, self.entities.size, len(self.all_messages),
        )
        self.allocation_log.append(allocation)

        # Update summary before building context
        recent_start = max(
            0, len(self.all_messages) - 10
        )
        self.summary.update_summary(recent_start)

        # Assign budgets to subsystems
        self.window.max_tokens = allocation["recent"]
        self.entities.max_tokens = allocation["entities"]
        self.vectors.max_tokens = allocation["retrieved"]

        # Build each section
        parts = []
        subsystem_tokens = {}

        # 1. System prompt
        if self.system_prompt:
            parts.append(f"System: {self.system_prompt}")

        # 2. Summary
        summary_text = self.summary.get_context()
        if summary_text:
            summary_text = summary_text[:allocation["summary"] * 4]
            parts.append(
                f"## Conversation Overview\n{summary_text}"
            )
            subsystem_tokens["summary"] = count_tokens(summary_text)
        else:
            subsystem_tokens["summary"] = 0

        # 3. Entities
        entity_text = self.entities.retrieve(
            query, max_tokens=allocation["entities"]
        )
        if entity_text:
            parts.append(f"## Known Facts\n{entity_text}")
            subsystem_tokens["entities"] = count_tokens(entity_text)
        else:
            subsystem_tokens["entities"] = 0

        # 4. Vector retrieval (exclude recent to avoid duplication)
        recent_msgs = self.window.get_context()
        recent_contents = set(m["content"] for m in recent_msgs)
        vector_results = self.vectors.retrieve(
            query, exclude_recent=recent_contents,
        )
        if vector_results:
            vec_text = self.vectors.format_results(vector_results)
            parts.append(
                f"## Relevant Past Context\n{vec_text}"
            )
            subsystem_tokens["retrieved"] = count_tokens(vec_text)
        else:
            subsystem_tokens["retrieved"] = 0

        # 5. Recent messages
        if recent_msgs:
            recent_text = "\n".join(
                f"{m['role']}: {m['content']}"
                for m in recent_msgs
            )
            parts.append(
                f"## Recent Conversation\n{recent_text}"
            )
            subsystem_tokens["recent"] = count_tokens(recent_text)
        else:
            subsystem_tokens["recent"] = 0

        context = "\n\n".join(parts)
        total_tokens = count_tokens(context)

        self.context_token_log.append(total_tokens)
        self.subsystem_token_log.append(subsystem_tokens)

        return {
            "context": context,
            "tokens": total_tokens,
            "allocation": allocation,
            "actual": subsystem_tokens,
            "vector_results": vector_results,
            "entity_text": entity_text,
            "summary_text": summary_text,
            "recent_count": len(recent_msgs),
        }

    def get_stats(self) -> Dict:
        return {
            "total_messages": len(self.all_messages),
            "total_turns": self.turn_count,
            "entities": self.entities.size,
            "vectors": self.vectors.size,
            "has_summary": bool(self.summary.summary),
            "total_tokens": sum(
                m["tokens"] for m in self.all_messages
            ),
        }

print("HybridMemory system defined!")

In [ ]:
#@title 🎧 Code Walkthrough: Test Hybrid
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_16_test_hybrid.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Let us test the hybrid system:

In [ ]:
hm = HybridMemory(
    total_tokens=8000,
    system_prompt="You are a helpful project assistant.",
    response_reserve=2000,
    recent_turns=5,
    vector_top_k=3,
)

# Full conversation
conversation = [
    ("user", "My name is Raj. Budget is $5,000. Deadline is March 15."),
    ("assistant", "Hello Raj! Budget of $5,000 with March 15 deadline. Got it."),
    ("user", "Team: Sarah handles frontend, Alex does backend, Priya works on ML."),
    ("assistant", "Great team of 4 with clear roles."),
    ("user", "We are using Python, FastAPI, React, and PostgreSQL."),
    ("assistant", "Solid stack. FastAPI + React + PostgreSQL."),
    ("user", "The chatbot needs returns handling and order tracking."),
    ("assistant", "Two core features: returns and order tracking."),
    ("user", "Deploy with Docker on Cloud Run. Need OAuth2 for auth."),
    ("assistant", "Docker + Cloud Run + OAuth2. Good choices."),
    ("user", "Use pytest with 80% coverage. Prometheus for monitoring."),
    ("assistant", "Testing and monitoring plan set."),
    ("user", "Sprint plan: Week 1 backend, Week 2 ML, Week 3 frontend, Week 4 testing."),
    ("assistant", "4-week sprint plan noted."),
    ("user", "I prefer concise answers and dark mode interfaces."),
    ("assistant", "Preferences noted. I will keep things concise."),
    ("user", "We had a meeting yesterday. Sarah raised concerns about the timeline."),
    ("assistant", "Sarah's timeline concerns noted. Shall we adjust?"),
    ("user", "Let us add a buffer week. New deadline: March 22."),
    ("assistant", "Deadline extended to March 22. Good call."),
]

for role, content in conversation:
    hm.add_message(role, content)

print(f"Hybrid Memory populated: {hm.get_stats()}")

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_17_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn

### TODO 1: Implement a Multi-Dimensional Evaluation

Build a function that evaluates the hybrid system across all four quality dimensions: factual recall, semantic recall, continuity, and cost efficiency.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_18_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def evaluate_hybrid(
    memory: HybridMemory,
    factual_tests: List[Dict],
    semantic_tests: List[Dict],
) -> Dict:
    """Evaluate hybrid memory across multiple dimensions.

    Args:
        memory: A populated HybridMemory instance
        factual_tests: List of dicts with:
            - 'query': The question
            - 'expected_keywords': Keywords that should
              appear in entity retrieval results
        semantic_tests: List of dicts with:
            - 'query': The question (phrased differently
              than original messages)
            - 'expected_keywords': Keywords that should
              appear in vector retrieval results

    Returns:
        Dict with keys:
            'factual_recall': fraction of factual tests
                where expected keywords found in entities
            'semantic_recall': fraction of semantic tests
                where expected keywords found in
                vector results
            'continuity': fraction of last 3 turns present
                in recent window
            'cost_efficiency': 1 - (avg context tokens /
                budget)
            'quality_score': weighted combination using
                weights [0.3, 0.3, 0.2, 0.2]
            'details': per-test breakdown

    Steps:
        1. For each factual test:
           a. Call memory.build_context(query)
           b. Check if expected_keywords appear in
              result['entity_text']
        2. For each semantic test:
           a. Call memory.build_context(query)
           b. Check if expected_keywords appear in
              the content of result['vector_results']
        3. Compute continuity: check if last 3 messages
           are in the recent window
        4. Compute cost efficiency from average context tokens
        5. Combine into quality_score

    Example:
        >>> result = evaluate_hybrid(hm, facts, semantics)
        >>> result['quality_score']
        0.82
    """
    # ---- YOUR CODE HERE ---- #

    results = {
        "factual_recall": 0.0,
        "semantic_recall": 0.0,
        "continuity": 0.0,
        "cost_efficiency": 0.0,
        "quality_score": 0.0,
        "details": [],
    }

    # TODO: Evaluate factual recall
    # TODO: Evaluate semantic recall
    # TODO: Compute continuity
    # TODO: Compute cost efficiency
    # TODO: Compute quality score

    return results

    # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_19_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Implement Budget Allocation Optimizer

Build a function that tests multiple allocation strategies and finds the one with the highest quality score.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_20_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def optimize_allocation(
    memory: HybridMemory,
    factual_tests: List[Dict],
    semantic_tests: List[Dict],
    n_trials: int = 20,
    seed: int = 42,
) -> Dict:
    """Find the best budget allocation via random search.

    Generates random allocation weights, evaluates each
    using evaluate_hybrid, and returns the best one.

    Args:
        memory: A populated HybridMemory instance
        factual_tests: Factual test cases
        semantic_tests: Semantic test cases
        n_trials: Number of random allocations to try
        seed: Random seed

    Returns:
        Dict with keys:
            'best_weights': Dict of optimal weights
            'best_score': The quality score achieved
            'all_trials': List of (weights, score) tuples
            'improvement': best_score - default_score

    Steps:
        1. Evaluate the default allocation as baseline
        2. For n_trials iterations:
           a. Generate random weights (4 values that sum to 1)
              using Dirichlet distribution:
              np.random.dirichlet([2, 1, 4, 6])
              for [summary, entities, retrieved, recent]
           b. Temporarily set the allocator weights
           c. Evaluate with evaluate_hybrid
           d. Track the best score and weights
        3. Return the best allocation

    Hints:
        - np.random.dirichlet generates weights that
          sum to 1
        - The alpha parameters [2, 1, 4, 6] bias toward
          reasonable allocations (more recent, less summary)
        - Remember to restore original allocator state
    """
    # ---- YOUR CODE HERE ---- #

    results = {
        "best_weights": {},
        "best_score": 0.0,
        "all_trials": [],
        "improvement": 0.0,
    }

    # TODO: Evaluate default allocation
    # TODO: Run random search
    # TODO: Return best allocation

    return results

    # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Code Walkthrough: Verify1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_21_verify1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Verification Cell for TODO 1

In [ ]:
# --- Test evaluate_hybrid ---
factual_tests = [
    {"query": "What is the budget?",
     "expected_keywords": ["5,000"]},
    {"query": "When is the deadline?",
     "expected_keywords": ["march"]},
    {"query": "What is the user's name?",
     "expected_keywords": ["raj"]},
    {"query": "How many people on the team?",
     "expected_keywords": ["4"]},
]

semantic_tests = [
    {"query": "How much money can we spend?",
     "expected_keywords": ["5,000"]},
    {"query": "Who is responsible for the ML work?",
     "expected_keywords": ["priya"]},
    {"query": "What are we using for deployment?",
     "expected_keywords": ["docker"]},
]

eval_result = evaluate_hybrid(hm, factual_tests, semantic_tests)

assert 0 <= eval_result["factual_recall"] <= 1
assert 0 <= eval_result["semantic_recall"] <= 1
assert 0 <= eval_result["continuity"] <= 1
assert 0 <= eval_result["cost_efficiency"] <= 1
assert 0 <= eval_result["quality_score"] <= 1

print(f"Factual recall:   {eval_result['factual_recall']:.0%}")
print(f"Semantic recall:  {eval_result['semantic_recall']:.0%}")
print(f"Continuity:       {eval_result['continuity']:.0%}")
print(f"Cost efficiency:  {eval_result['cost_efficiency']:.0%}")
print(f"Quality score:    {eval_result['quality_score']:.3f}")

print("\nAll evaluate_hybrid tests passed!")

In [ ]:
#@title 🎧 Code Walkthrough: Verify2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_22_verify2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Verification Cell for TODO 2

In [ ]:
# --- Test optimize_allocation ---
opt_result = optimize_allocation(
    hm, factual_tests, semantic_tests,
    n_trials=10, seed=42,
)

assert "best_weights" in opt_result
assert "best_score" in opt_result
assert len(opt_result["all_trials"]) == 10
assert abs(sum(opt_result["best_weights"].values()) - 1.0) < 0.05

print(f"Best score: {opt_result['best_score']:.3f}")
print(f"Best weights: {opt_result['best_weights']}")
print(f"Improvement over default: {opt_result['improvement']:+.3f}")

print("\nAll optimize_allocation tests passed!")

In [ ]:
#@title 🎧 Listen: Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_23_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


<details>
<summary><b>Solution: evaluate_hybrid</b></summary>

In [ ]:
def evaluate_hybrid(memory, factual_tests, semantic_tests):
    details = []

    # Factual recall
    fact_hits = 0
    for test in factual_tests:
        ctx = memory.build_context(test["query"])
        entity_text = (ctx.get("entity_text", "") or "").lower()
        found = any(kw.lower() in entity_text for kw in test["expected_keywords"])
        fact_hits += found
        details.append({"type": "factual", "query": test["query"], "hit": found})
    factual_recall = fact_hits / len(factual_tests) if factual_tests else 0

    # Semantic recall
    sem_hits = 0
    for test in semantic_tests:
        ctx = memory.build_context(test["query"])
        vec_text = ""
        if ctx.get("vector_results"):
            vec_text = " ".join(
                msg["content"].lower() for msg, _ in ctx["vector_results"]
            )
        found = any(kw.lower() in vec_text for kw in test["expected_keywords"])
        sem_hits += found
        details.append({"type": "semantic", "query": test["query"], "hit": found})
    semantic_recall = sem_hits / len(semantic_tests) if semantic_tests else 0

    # Continuity
    last_3 = memory.all_messages[-3:]
    recent = memory.window.get_context()
    recent_contents = set(m["content"] for m in recent)
    cont_hits = sum(1 for m in last_3 if m["content"] in recent_contents)
    continuity = cont_hits / len(last_3) if last_3 else 0

    # Cost efficiency
    avg_tokens = np.mean(memory.context_token_log) if memory.context_token_log else 0
    cost_efficiency = 1 - (avg_tokens / memory.allocator.budget) if memory.allocator.budget > 0 else 0
    cost_efficiency = max(0, min(1, cost_efficiency))

    # Quality score
    quality = (0.3 * factual_recall + 0.3 * semantic_recall
               + 0.2 * continuity + 0.2 * cost_efficiency)

    return {
        "factual_recall": factual_recall,
        "semantic_recall": semantic_recall,
        "continuity": continuity,
        "cost_efficiency": cost_efficiency,
        "quality_score": quality,
        "details": details,
    }

</details>

<details>
<summary><b>Solution: optimize_allocation</b></summary>

In [ ]:
def optimize_allocation(memory, factual_tests, semantic_tests, n_trials=20, seed=42):
    np.random.seed(seed)
    keys = ["summary", "entities", "retrieved", "recent"]

    # Baseline
    baseline = evaluate_hybrid(memory, factual_tests, semantic_tests)
    baseline_score = baseline["quality_score"]

    best_score = baseline_score
    best_weights = {"summary": 0.06, "entities": 0.04, "retrieved": 0.36, "recent": 0.54}
    all_trials = []

    for _ in range(n_trials):
        raw = np.random.dirichlet([2, 1, 4, 6])
        weights = dict(zip(keys, raw))

        old_alloc = memory.allocator.allocate_fixed
        def patched_alloc(w=None):
            return {k: int(memory.allocator.budget * weights[k]) for k in keys}
        memory.allocator.allocate_adaptive = lambda q, ne, nm, w=weights: {
            k: int(memory.allocator.budget * w[k]) for k in keys
        }

        score = evaluate_hybrid(memory, factual_tests, semantic_tests)["quality_score"]
        all_trials.append((dict(weights), score))

        if score > best_score:
            best_score = score
            best_weights = dict(weights)

    # Restore
    memory.allocator.allocate_adaptive = BudgetAllocator.allocate_adaptive.__get__(memory.allocator)

    return {
        "best_weights": best_weights,
        "best_score": best_score,
        "all_trials": all_trials,
        "improvement": best_score - baseline_score,
    }

</details>

In [ ]:
#@title 🎧 Code Walkthrough: Load Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_24_load_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Load solutions ---
def evaluate_hybrid(memory, factual_tests, semantic_tests):
    details = []
    fact_hits = 0
    for test in factual_tests:
        ctx = memory.build_context(test["query"])
        entity_text = (ctx.get("entity_text", "") or "").lower()
        found = any(kw.lower() in entity_text for kw in test["expected_keywords"])
        fact_hits += found
        details.append({"type": "factual", "query": test["query"], "hit": found})
    factual_recall = fact_hits / len(factual_tests) if factual_tests else 0
    sem_hits = 0
    for test in semantic_tests:
        ctx = memory.build_context(test["query"])
        vec_text = ""
        if ctx.get("vector_results"):
            vec_text = " ".join(msg["content"].lower() for msg, _ in ctx["vector_results"])
        found = any(kw.lower() in vec_text for kw in test["expected_keywords"])
        sem_hits += found
        details.append({"type": "semantic", "query": test["query"], "hit": found})
    semantic_recall = sem_hits / len(semantic_tests) if semantic_tests else 0
    last_3 = memory.all_messages[-3:]
    recent = memory.window.get_context()
    recent_contents = set(m["content"] for m in recent)
    cont_hits = sum(1 for m in last_3 if m["content"] in recent_contents)
    continuity = cont_hits / len(last_3) if last_3 else 0
    avg_tokens = np.mean(memory.context_token_log) if memory.context_token_log else 0
    cost_efficiency = max(0, min(1, 1 - (avg_tokens / memory.allocator.budget)))
    quality = 0.3 * factual_recall + 0.3 * semantic_recall + 0.2 * continuity + 0.2 * cost_efficiency
    return {
        "factual_recall": factual_recall, "semantic_recall": semantic_recall,
        "continuity": continuity, "cost_efficiency": cost_efficiency,
        "quality_score": quality, "details": details,
    }

print("Solutions loaded!")

In [ ]:
#@title 🎧 Listen: Putting Together Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_25_putting_together_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Putting It All Together

Let us run the full hybrid system and compare it against each individual memory type.

In [ ]:
#@title 🎧 Code Walkthrough: Individual Systems Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_26_individual_systems_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Rebuild individual memory systems for fair comparison
# (same conversation, same queries)

# --- Individual systems ---
from collections import deque

class IndividualWindow:
    """Pure sliding window for comparison."""
    def __init__(self):
        self.messages = []
    def add(self, role, content):
        self.messages.append({"role": role, "content": content,
                              "tokens": count_tokens(content)})
    def query(self, q, budget=5500):
        result = []
        total = 0
        for m in reversed(self.messages):
            if total + m["tokens"] > budget:
                break
            result.insert(0, m)
            total += m["tokens"]
        return " ".join(m["content"] for m in result)

class IndividualEntity:
    """Pure entity memory for comparison."""
    def __init__(self):
        self.store = EntityStore(max_tokens=5500)
        self.turn = 0
    def add(self, role, content):
        if role == "user":
            self.turn += 1
        self.store.extract_and_store(content, self.turn)
    def query(self, q):
        return self.store.retrieve(q, max_tokens=5500)

class IndividualVector:
    """Pure vector memory for comparison."""
    def __init__(self):
        self.vm = VectorMemory(max_tokens=5500, top_k=10)
    def add(self, role, content):
        self.vm.add(content, role)
    def query(self, q):
        results = self.vm.retrieve(q)
        return " ".join(m["content"] for m, _ in results)

# Build all systems
iw = IndividualWindow()
ie = IndividualEntity()
iv = IndividualVector()

for role, content in conversation:
    iw.add(role, content)
    ie.add(role, content)
    iv.add(role, content)

print("All individual systems built for comparison.")

In [ ]:
#@title 🎧 Code Walkthrough: Comparison Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_27_comparison_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Head-to-head comparison
test_queries = [
    ("What is the budget?", ["5,000"]),
    ("Who is on the team?", ["sarah", "alex", "priya"]),
    ("When is the deadline?", ["march"]),
    ("How much can we spend?", ["5,000"]),
    ("What are we using for deployment?", ["docker"]),
    ("What does Sarah do?", ["frontend"]),
    ("What monitoring tools?", ["prometheus"]),
    ("What are the core features?", ["returns", "order"]),
]

print("=" * 70)
print("HEAD-TO-HEAD: HYBRID vs INDIVIDUAL MEMORY SYSTEMS")
print("=" * 70)
print(f"\n{'Query':<35} {'Window':>7} {'Entity':>7} "
      f"{'Vector':>7} {'Hybrid':>7}")
print("-" * 68)

scores = {"window": 0, "entity": 0, "vector": 0, "hybrid": 0}

for query, keywords in test_queries:
    # Window
    w_text = iw.query(query).lower()
    w_hit = any(kw in w_text for kw in keywords)

    # Entity
    e_text = ie.query(query).lower()
    e_hit = any(kw in e_text for kw in keywords)

    # Vector
    v_text = iv.query(query).lower()
    v_hit = any(kw in v_text for kw in keywords)

    # Hybrid
    h_ctx = hm.build_context(query)
    h_text = h_ctx["context"].lower()
    h_hit = any(kw in h_text for kw in keywords)

    scores["window"] += w_hit
    scores["entity"] += e_hit
    scores["vector"] += v_hit
    scores["hybrid"] += h_hit

    w_mark = "HIT" if w_hit else "miss"
    e_mark = "HIT" if e_hit else "miss"
    v_mark = "HIT" if v_hit else "miss"
    h_mark = "HIT" if h_hit else "miss"

    print(f"  {query:<33} {w_mark:>7} {e_mark:>7} "
          f"{v_mark:>7} {h_mark:>7}")

n = len(test_queries)
print("-" * 68)
print(f"  {'ACCURACY':<33} "
      f"{scores['window']/n:>6.0%} {scores['entity']/n:>6.0%} "
      f"{scores['vector']/n:>6.0%} {scores['hybrid']/n:>6.0%}")

In [ ]:
#@title 🎧 Listen: Results Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_28_results_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Training and Results

### Visualization 1: Multi-Dimensional Quality Comparison

In [ ]:
#@title 🎧 What to Look For: Vis1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_29_vis1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
eval_result = evaluate_hybrid(hm, factual_tests, semantic_tests)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Radar chart (simplified as bar chart)
ax1 = axes[0]
dimensions = ["Factual\nRecall", "Semantic\nRecall",
              "Continuity", "Cost\nEfficiency"]
hybrid_scores = [
    eval_result["factual_recall"],
    eval_result["semantic_recall"],
    eval_result["continuity"],
    eval_result["cost_efficiency"],
]

# Approximate scores for individual systems
window_scores = [0.3, 0.2, 1.0, 0.6]
entity_scores = [0.9, 0.3, 0.0, 0.85]
vector_scores = [0.5, 0.9, 0.0, 0.7]

x = np.arange(len(dimensions))
width = 0.2

ax1.bar(x - 1.5*width, window_scores, width,
        label="Window", color="#10B981", alpha=0.8)
ax1.bar(x - 0.5*width, entity_scores, width,
        label="Entity", color="#F59E0B", alpha=0.8)
ax1.bar(x + 0.5*width, vector_scores, width,
        label="Vector", color="#8B5CF6", alpha=0.8)
ax1.bar(x + 1.5*width, hybrid_scores, width,
        label="Hybrid", color="#3B82F6", alpha=0.8)

ax1.set_xticks(x)
ax1.set_xticklabels(dimensions, fontsize=10)
ax1.set_ylabel("Score (0-1)", fontsize=12)
ax1.set_title("Quality Dimensions: Hybrid vs Individual",
              fontsize=14, fontweight="bold")
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1.15)
ax1.grid(True, alpha=0.3, axis="y")

# Right: Overall quality scores
ax2 = axes[1]
systems = ["Window\nOnly", "Entity\nOnly", "Vector\nOnly", "HYBRID"]
overall = [
    0.3*0.3 + 0.3*0.2 + 0.2*1.0 + 0.2*0.6,
    0.3*0.9 + 0.3*0.3 + 0.2*0.0 + 0.2*0.85,
    0.3*0.5 + 0.3*0.9 + 0.2*0.0 + 0.2*0.7,
    eval_result["quality_score"],
]
colors = ["#10B981", "#F59E0B", "#8B5CF6", "#3B82F6"]

bars = ax2.bar(systems, overall, color=colors, width=0.5,
               edgecolor="white")
for bar, score in zip(bars, overall):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f"{score:.3f}", ha="center", fontsize=13,
             fontweight="bold")

ax2.set_ylabel("Composite Quality Score", fontsize=12)
ax2.set_title("Overall Quality: Hybrid Wins",
              fontsize=14, fontweight="bold")
ax2.set_ylim(0, 1.0)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("hybrid_quality.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
#@title 🎧 Listen: Vis2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_30_vis2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualization 2: Token Budget Allocation

In [ ]:
#@title 🎧 What to Look For: Vis2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_31_vis2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Show how budget was allocated across queries
if hm.subsystem_token_log:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Stacked bar of token allocation per query
    ax1 = axes[0]
    n_queries = min(len(hm.subsystem_token_log), 8)
    query_labels = [f"Q{i+1}" for i in range(n_queries)]

    bottoms = np.zeros(n_queries)
    component_colors = {
        "summary": "#F59E0B",
        "entities": "#10B981",
        "retrieved": "#8B5CF6",
        "recent": "#3B82F6",
    }

    for component, color in component_colors.items():
        values = [
            hm.subsystem_token_log[i].get(component, 0)
            for i in range(n_queries)
        ]
        ax1.bar(query_labels, values, bottom=bottoms,
                color=color, label=component.capitalize(),
                edgecolor="white", linewidth=0.5)
        bottoms += values

    ax1.set_xlabel("Query", fontsize=12)
    ax1.set_ylabel("Tokens", fontsize=12)
    ax1.set_title("Token Allocation Per Query",
                  fontsize=14, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3, axis="y")

    # Right: Average allocation pie chart
    ax2 = axes[1]
    avg_alloc = defaultdict(float)
    for log in hm.subsystem_token_log[:n_queries]:
        for k, v in log.items():
            avg_alloc[k] += v
    for k in avg_alloc:
        avg_alloc[k] /= n_queries

    labels = list(avg_alloc.keys())
    sizes = list(avg_alloc.values())
    colors_pie = [component_colors.get(l, "#999") for l in labels]

    wedges, texts, autotexts = ax2.pie(
        sizes, labels=[l.capitalize() for l in labels],
        autopct="%1.0f%%", colors=colors_pie,
        textprops={"fontsize": 11},
    )
    for at in autotexts:
        at.set_fontweight("bold")
    ax2.set_title("Average Budget Allocation",
                  fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig("budget_allocation.png", dpi=150,
                bbox_inches="tight")
    plt.show()

In [ ]:
#@title 🎧 Listen: Vis3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_32_vis3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualization 3: Hybrid vs Individual — Accuracy Across Query Types

In [ ]:
#@title 🎧 What to Look For: Vis3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_33_vis3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Categorize test queries
query_categories = {
    "Factual": [
        ("What is the budget?", ["5,000"]),
        ("When is the deadline?", ["march"]),
        ("What is the user's name?", ["raj"]),
    ],
    "Semantic": [
        ("How much can we spend?", ["5,000"]),
        ("Who handles ML?", ["priya"]),
        ("What are we using for hosting?", ["docker", "cloud"]),
    ],
    "Continuity": [
        ("What did Sarah say about the timeline?", ["sarah", "timeline"]),
        ("What about the new deadline?", ["march 22"]),
    ],
}

fig, ax = plt.subplots(figsize=(12, 6))

categories = list(query_categories.keys())
x = np.arange(len(categories))
width = 0.2

system_results = {
    "Window": [], "Entity": [], "Vector": [], "Hybrid": [],
}

for cat, queries in query_categories.items():
    for sys_name, sys_fn in [
        ("Window", lambda q: iw.query(q).lower()),
        ("Entity", lambda q: ie.query(q).lower()),
        ("Vector", lambda q: iv.query(q).lower()),
        ("Hybrid", lambda q: hm.build_context(q)["context"].lower()),
    ]:
        hits = sum(
            any(kw in sys_fn(q) for kw in kws)
            for q, kws in queries
        )
        system_results[sys_name].append(
            hits / len(queries) if queries else 0
        )

sys_colors = {
    "Window": "#10B981", "Entity": "#F59E0B",
    "Vector": "#8B5CF6", "Hybrid": "#3B82F6",
}

for i, (sys_name, scores_list) in enumerate(system_results.items()):
    ax.bar(x + i * width, scores_list, width,
           label=sys_name, color=sys_colors[sys_name],
           alpha=0.8)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Accuracy by Query Type: Hybrid Excels Everywhere",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("accuracy_by_type.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
#@title 🎧 Listen: Final Output Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_34_final_output_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Final Output

Let us run a complete production-style demonstration. We will simulate a new session where the user asks varied questions, and the hybrid system uses all four memory subsystems to provide comprehensive answers.

In [ ]:
#@title 🎧 Code Walkthrough: Final Output Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_35_final_output_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 65)
print("  FINAL DEMO: PRODUCTION-READY HYBRID MEMORY SYSTEM")
print("=" * 65)

demo_queries = [
    "What is my budget and when is the deadline?",
    "Give me an overview of the entire project.",
    "Who is responsible for the ML pipeline?",
    "How much can I spend on cloud infrastructure?",
    "What was the concern Sarah raised?",
    "Can you elaborate on the deployment plan?",
    "What are all the technologies we are using?",
]

print()
for query in demo_queries:
    ctx = hm.build_context(query)

    print(f"User: \"{query}\"")
    print(f"  Allocation: ", end="")
    for k, v in ctx["actual"].items():
        if v > 0:
            print(f"{k}={v}tok ", end="")
    print(f"  (total: {ctx['tokens']} tokens)")

    # Show what each subsystem contributed
    contributions = []
    if ctx["summary_text"]:
        contributions.append(f"Summary: {ctx['summary_text'][:50]}...")
    if ctx["entity_text"]:
        entities = ctx["entity_text"].split("\n")[:2]
        contributions.append(f"Entities: {'; '.join(e.strip() for e in entities)}")
    if ctx["vector_results"]:
        top_msg = ctx["vector_results"][0][0]["content"][:50]
        contributions.append(f"Vector: [{ctx['vector_results'][0][1]:.2f}] {top_msg}...")
    contributions.append(f"Recent: {ctx['recent_count']} messages")

    for c in contributions:
        print(f"    {c}")
    print()

# Final statistics
stats = hm.get_stats()
print("=" * 65)
print(f"  System Statistics:")
print(f"    Total messages:    {stats['total_messages']}")
print(f"    Total turns:       {stats['total_turns']}")
print(f"    Entities stored:   {stats['entities']}")
print(f"    Vectors indexed:   {stats['vectors']}")
print(f"    Has summary:       {stats['has_summary']}")
print(f"    Total conv tokens: {stats['total_tokens']}")
if hm.context_token_log:
    avg_ctx = np.mean(hm.context_token_log)
    print(f"    Avg context tokens: {avg_ctx:.0f}")
    print(f"    Budget utilization: {avg_ctx/hm.allocator.budget:.0%}")
print(f"")
print(f"  The hybrid system combines four memory types:")
print(f"    - Sliding window for conversation flow")
print(f"    - Summary for the big picture")
print(f"    - Entity store for precise facts")
print(f"    - Vector index for semantic retrieval")
print(f"")
print(f"  Together, they provide comprehensive, accurate,")
print(f"  and cost-efficient context for any query type.")
print("=" * 65)

In [ ]:
#@title 🎧 Listen: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_36_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### What Did We Build?

Over the course of five notebooks, we built a complete memory architecture toolkit:

1. **Notebook 1: Buffer + Sliding Window** — the foundations. Buffer is simple but unbounded. Sliding window is bounded but forgetful.

2. **Notebook 2: Summary Memory** — compressing old messages preserves themes but loses specific details. The compression ratio is typically 20-30:1.

3. **Notebook 3: Entity Memory** — extracting structured facts preserves precision (exact numbers, names, dates) but misses general context.

4. **Notebook 4: Vector Memory** — embedding-based retrieval captures semantic similarity, allowing "How much can I spend?" to find the budget message even without keyword overlap.

5. **Notebook 5: Hybrid Memory** — combining all four systems with adaptive budget allocation achieves the highest quality across all dimensions.

### The Design Decision Framework

When choosing a memory architecture for your application:

| Scenario | Best Architecture |
|----------|-------------------|
| Short conversations (<20 turns) | Buffer or sliding window |
| Simple chatbot | Sliding window (K=10) |
| Fact-heavy application (CRM, support) | Entity memory + window |
| Long conversations, broad topics | Vector memory + window |
| Production assistant | Full hybrid |

### Questions to Think About

1. We used fixed allocation weights with simple adaptive rules. How would you use the conversation itself to dynamically adjust weights? Could the LLM decide its own allocation?

2. What about **multi-session memory** — remembering facts across completely separate conversations? How would you persist the entity store and vector index between sessions?

3. Our evaluation uses keyword matching. In production, you would use an LLM to judge whether the context was sufficient to answer the query correctly. How would you set up that evaluation pipeline?

4. **Memory decay**: should old memories fade over time? Or should the system remember everything forever? What are the privacy implications?

5. The hybrid system has four subsystems, each with its own parameters (window size, top-k, summary length, entity budget). How would you tune all of these simultaneously for a specific application?

### Production Considerations

If you were deploying this system, you would also need:
- **Persistence**: Store entities and vectors in a database (PostgreSQL + pgvector, or Pinecone/Weaviate)
- **Async processing**: Entity extraction and summarization should happen asynchronously, not blocking the response
- **Cache**: Cache embeddings to avoid recomputing them
- **Monitoring**: Track retrieval quality, context token usage, and user satisfaction
- **Privacy**: Allow users to delete specific memories ("forget that I told you about X")
- **Rate limiting**: Summarization calls to the LLM should be rate-limited

### Closing Thought

Memory architecture is, at its core, **context engineering**. Every approach we have built is ultimately about the same thing: filling the LLM's context window with the *right* information at the *right* time. The "right" approach depends entirely on what your application needs to remember and how much context budget you have.

The hybrid system gives you the most flexibility, but it also has the most moving parts. Start simple — a sliding window is often enough. Add complexity only when you measure a specific failure mode (forgotten facts, missed callbacks, budget overflow). Let the data guide your architecture.

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_37_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 60)
print("  COURSE COMPLETE!")
print("=" * 60)
print()
print("  You have built 5 memory systems from scratch:")
print("    1. Conversation Buffer")
print("    2. Sliding Window")
print("    3. Summary Memory")
print("    4. Entity Memory")
print("    5. Vector-Backed Memory")
print("    6. Hybrid Memory (combining all 5)")
print()
print("  Key equations:")
print("    Buffer:  C(n) = n * t_avg              (unbounded)")
print("    Window:  C = K * t_avg                  (bounded)")
print("    Summary: C = t_summary + K * t_avg      (bounded)")
print("    Entity:  C = t_entities + K * t_avg     (bounded)")
print("    Vector:  C = k * t_retrieved + K * t_avg (bounded)")
print("    Hybrid:  C = t_sum + t_ent + t_ret + t_recent (bounded)")
print()
print("  The fundamental insight:")
print("  Memory architecture is context engineering.")
print("  Fill the window with the right information,")
print("  at the right time, within the right budget.")
print()
print("  Congratulations on completing this series!")